# Schwinger hadron challenge — Parts 2 & 3: engineering and mitigation

Ex 2.2 gate accounting, Ex 2.3 `select_chain`, Ex 3.1 ODR, Ex 3.2 `mitigation_options`. Offline against `FakeKingston` (needs `qiskit-ibm-runtime`). Imports the Part 0 answers from `schwinger_core.py`.

Not covered here (not solved yet): Ex 2.1 cell code beyond the layout, `evolve_circuits_matched`, Ex 2.4 flight plan, Ex 3.3 rehearsal, Part 4 hardware.

In [1]:
import math
import time
import copy
import warnings
import numpy as np
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeKingston

from schwinger_core import *
REF_DIR = 'reference_data'
L, t = 34, 8

## Ex 2.1 / 2.2 — layout and gate accounting

In [2]:
qc_wave_init = prep_wave(L, vacuum_prep_theta_OV_1, vacuum_prep_theta_OV_3, wave_prep_theta_O_11, wave_prep_theta_O_22)
qc_wave, qc_wave_mitig = evolve_circuits(qc_wave_init, L, t, m, g)

backend = FakeKingston()
qc_isa = generate_preset_pass_manager(optimization_level=3, backend=backend, seed_transpiler=42).run(qc_wave)
layout = list(qc_isa.layout.initial_index_layout(filter_ancillas=True))
pm_pinned = generate_preset_pass_manager(optimization_level=3, backend=backend, seed_transpiler=42,
                                         initial_layout=layout, layout_method='trivial')
isa_physics = pm_pinned.run(qc_wave)
print('swap-free:', layout == list(qc_isa.layout.final_index_layout()))

def count_2q(qc: QuantumCircuit, reps: int = 4) -> int:
    return sum(1 for i in qc.decompose(reps=reps).data
               if len(i.qubits) > 1 and not getattr(i.operation, '_directive', False))

n2q_logical = count_2q(qc_wave)
n_cz_physics = int(isa_physics.count_ops().get('cz', 0))
print(f'{n2q_logical} logical 2q gates -> {n_cz_physics} CZ ({100 * (1 - n_cz_physics / n2q_logical):.1f} % fewer)')
print('\n--- paste into the grader ---')
print(f'n2q_logical = {n2q_logical}')
print(f'n_cz_physics = {n_cz_physics}')

swap-free: True


5758 logical 2q gates -> 4958 CZ (13.9 % fewer)

--- paste into the grader ---
n2q_logical = 5758
n_cz_physics = 4958


## Ex 2.3 — calibration-aware chain selection

In [3]:
def target_error_tables(backend, dead: float = 0.5):
    """(cz, ro, sx) error dictionaries from backend.target; missing values count as 1.0 (dead). [given]"""
    tg = backend.target
    def err(props):
        e = getattr(props, "error", None) if props is not None else None
        return 1.0 if e is None or (isinstance(e, float) and math.isnan(e)) else float(e)
    cz = {}
    for q, p in tg["cz"].items():
        key = tuple(sorted(q))
        cz[key] = max(cz.get(key, 0.0), err(p))
    ro = {q[0]: err(p) for q, p in tg["measure"].items()}
    sx = {q[0]: err(p) for q, p in tg["sx"].items()}
    return cz, ro, sx


def _nl(e: float) -> float:
    return -math.log(max(1e-12, 1.0 - min(e, 0.999)))


def chain_cost(chain, backend, n_cz_per_bond: float = 74.0, n_sx_per_qubit: float = 150.0) -> float:
    """Expected number of errors on the circuit (log form) for a given chain of physical qubits. [given]"""
    cz, ro, sx = target_error_tables(backend)
    c = sum(n_cz_per_bond * _nl(cz[tuple(sorted((a, b)))]) for a, b in zip(chain[:-1], chain[1:]))
    c += sum(_nl(ro[q]) + n_sx_per_qubit * _nl(sx[q]) for q in chain)
    return float(c)


def select_chain(backend, n_qubits: int = 68, time_budget: float = 15.0, seed: int = 0,
                 n_cz_per_bond: float = 74.0, n_sx_per_qubit: float = 150.0, dead: float = 0.5, exclude=()) -> list[int]:
    """Randomised DFS with restarts over the coupling graph (dead edges/qubits and `exclude`d qubits removed);
    returns the cheapest path found within time_budget seconds."""
    cz, ro, sx = target_error_tables(backend)
    exclude = set(int(q) for q in exclude)
    # BEGIN ANSWER
    rng = np.random.default_rng(seed)
    t_end = time.time() + time_budget
    alive = {q for q in ro if ro[q] < dead and sx.get(q, 1.0) < dead and q not in exclude}
    node_cost = {q: _nl(ro[q]) + n_sx_per_qubit * _nl(sx.get(q, 1.0)) for q in alive}
    adj = {q: {} for q in alive}                      # adj[a][b] = edge cost
    for (a, b), e in cz.items():
        if e < dead and a in alive and b in alive:
            adj[a][b] = adj[b][a] = n_cz_per_bond * _nl(e)

    def reachable(start, visited, limit):
        """Number of unvisited qubits reachable from start (stops counting at limit)."""
        seen, stack = {start}, [start]
        while stack and len(seen) < limit:
            for nb in adj[stack.pop()]:
                if nb not in seen and nb not in visited:
                    seen.add(nb)
                    stack.append(nb)
        return len(seen)

    best, best_cost = None, math.inf
    starts = sorted(alive)
    while time.time() < t_end:
        start = starts[rng.integers(len(starts))]
        path, visited = [start], {start}
        cost = node_cost[start]
        expansions = [0]
        noise = rng.uniform(0.0, 0.5)             # exploration strength for this restart

        def dfs(cost):
            nonlocal best, best_cost
            if cost >= best_cost or time.time() > t_end or expansions[0] > 5000:
                return
            expansions[0] += 1
            if len(path) == n_qubits:
                best, best_cost = list(path), cost
                return
            u = path[-1]
            nbrs = [v for v in adj[u] if v not in visited]
            scale = np.mean([adj[u][v] + node_cost[v] for v in nbrs]) if nbrs else 0.0
            nbrs.sort(key=lambda v: adj[u][v] + node_cost[v] + noise * scale * rng.random())
            for v in nbrs:
                if reachable(v, visited, n_qubits - len(path)) < n_qubits - len(path):
                    continue                          # dead end: not enough qubits left beyond v
                path.append(v); visited.add(v)
                dfs(cost + adj[u][v] + node_cost[v])
                path.pop(); visited.discard(v)

        dfs(cost)
    # END ANSWER
    assert best is not None and len(best) == n_qubits and len(set(best)) == n_qubits, "no valid chain found"
    return [int(q) for q in best]

In [4]:
n_cz_per_bond = n_cz_physics / (2 * L - 1)
n_sx_per_qubit = isa_physics.count_ops().get('sx', 0) / (2 * L)
t0_ = time.time()
chain = select_chain(backend, n_qubits=2 * L, time_budget=15.0, seed=0, n_cz_per_bond=n_cz_per_bond, n_sx_per_qubit=n_sx_per_qubit)
print(f'select_chain: {time.time() - t0_:.1f} s;  chain = {chain}')
_edges = {tuple(sorted(e)) for e in backend.coupling_map.get_edges()}
assert all(tuple(sorted((a, b))) in _edges for a, b in zip(chain[:-1], chain[1:])) and len(set(chain)) == 2 * L
c_tr, c_me = chain_cost(layout, backend, n_cz_per_bond, n_sx_per_qubit), chain_cost(chain, backend, n_cz_per_bond, n_sx_per_qubit)
print(f'expected errors: transpiler layout {c_tr:.2f} | selected chain {c_me:.2f} ({100 * (1 - c_me / c_tr):+.1f} % fewer)')

select_chain: 15.0 s;  chain = [140, 141, 142, 143, 136, 123, 124, 125, 126, 127, 128, 129, 118, 109, 108, 107, 97, 87, 88, 89, 90, 91, 92, 93, 79, 73, 74, 75, 59, 55, 54, 53, 39, 33, 34, 35, 19, 15, 14, 13, 12, 11, 18, 31, 30, 29, 28, 27, 26, 25, 37, 45, 46, 47, 48, 49, 50, 51, 58, 71, 70, 69, 68, 67, 66, 65, 64, 63]
expected errors: transpiler layout 13.38 | selected chain 12.98 (+2.9 % fewer)


## Ex 3.1 — ODR, its uncertainty and its bias

In [5]:
def odr_mitigate(chi, chi_cal, chi_exact, L, suppression_threshold=0.01):
    """Operator decoherence renormalization with mirror-pooled factors and post-selection.

    Works on a single profile (2L,) or on a stack of profiles (n, 2L) at once (used by odr_uncertainty)."""
    chi = np.asarray(chi, dtype=float)
    chi_cal = np.asarray(chi_cal, dtype=float)
    chi_exact = np.asarray(chi_exact, dtype=float)
    single = chi.ndim == 1
    chi2 = np.atleast_2d(chi)
    f = np.broadcast_to((1 - chi_cal) / (1 - chi_exact), chi2.shape)
    q = np.arange(L)
    idx = np.stack([q, 2 * L - 1 - q])                         # (2, L): site and its mirror
    chi_pair, f_pair = chi2[:, idx], f[:, idx]                  # (n, 2, L)
    keep = f_pair > suppression_threshold
    n_keep = keep.sum(axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        chi_mean = np.where(keep, chi_pair, 0.0).sum(axis=1) / n_keep
        f_mean = np.where(keep, f_pair, 0.0).sum(axis=1) / n_keep
        half = 1 - (1 - chi_mean) / f_mean
    half = np.where(n_keep > 0, half, np.nan)
    out = np.concatenate([half, half[:, ::-1]], axis=1)         # mirror-symmetric output
    return out[0] if single else out


def odr_uncertainty(chi, chi_std, chi_cal, chi_cal_std, chi_exact, L, suppression_threshold=0.01,
                    n_samples=2000, seed=0):
    """1-sigma uncertainty of odr_mitigate(...) by Monte-Carlo propagation of the Estimator stds."""
    import warnings
    rng = np.random.default_rng(seed)
    chi = np.asarray(chi, dtype=float)
    chi_cal = np.asarray(chi_cal, dtype=float)
    chi_s = rng.normal(chi, np.asarray(chi_std, dtype=float), size=(n_samples, chi.size))
    cal_s = rng.normal(chi_cal, np.asarray(chi_cal_std, dtype=float), size=(n_samples, chi_cal.size))
    samples = odr_mitigate(chi_s, cal_s, chi_exact, L, suppression_threshold)
    with np.errstate(invalid="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        sigma = np.nanstd(samples, axis=0, ddof=1)
    central = odr_mitigate(chi, chi_cal, chi_exact, L, suppression_threshold)
    return np.where(np.isfinite(central), sigma, np.nan)


def odr_bias(chi_true, f_phys, f_cal):
    """Systematic shift of the ODR estimate when the physics-circuit factor differs from the calibration factor."""
    return (1 - np.asarray(chi_true, dtype=float)) * (1 - np.asarray(f_phys, dtype=float) / np.asarray(f_cal, dtype=float))

In [6]:
# the notebook's self-test on synthetic data
import challenge_utils as cu
cp_average = lambda c: 0.5 * (np.asarray(c, float) + np.asarray(c, float)[::-1])
rng = np.random.default_rng(2026)
chi_true_syn = cp_average(np.loadtxt(f'{REF_DIR}/chi_wave_evolved_sim_L34_maxbond40.txt'))
chi_exact_syn = np.loadtxt(f'{REF_DIR}/chi_wave_t0_sim_L34.txt')
f_cal_syn = rng.uniform(0.15, 0.6, 2 * L)
f_cal_syn[[5, 2 * L - 1 - 5]] = 0.005
chi_meas_syn = 1 - f_cal_syn * (1 - chi_true_syn)
chi_cal_syn = 1 - f_cal_syn * (1 - chi_exact_syn)

rec = odr_mitigate(chi_meas_syn, chi_cal_syn, chi_exact_syn, L)
ok = np.isfinite(rec)
assert not ok[5] and not ok[2 * L - 6] and ok.sum() == 2 * L - 2 and np.max(np.abs(rec[ok] - chi_true_syn[ok])) < 1e-10
print(f'(1) noise-free recovery: max |error| = {np.max(np.abs(rec[ok] - chi_true_syn[ok])):.1e}, NaN sites = {np.where(~ok)[0].tolist()}')

rec_b = odr_mitigate(1 - 0.9 * f_cal_syn * (1 - chi_true_syn), chi_cal_syn, chi_exact_syn, L)
predicted = odr_bias(chi_true_syn, 0.9 * f_cal_syn, f_cal_syn)
assert np.nanmax(np.abs((rec_b - chi_true_syn) - predicted)[ok]) < 1e-10
print(f'(2) bias formula exact to {np.nanmax(np.abs((rec_b - chi_true_syn) - predicted)[ok]):.1e}')

std_syn = np.full(2 * L, 0.01)
sigma_mc = odr_uncertainty(chi_meas_syn, std_syn, chi_cal_syn, std_syn, chi_exact_syn, L, n_samples=2000, seed=0)
reps = np.array([odr_mitigate(rng.normal(chi_meas_syn, std_syn), rng.normal(chi_cal_syn, std_syn), chi_exact_syn, L) for _ in range(300)])
ratio = sigma_mc[ok] / np.nanstd(reps, axis=0, ddof=1)[ok]
assert 0.8 < np.median(ratio) < 1.25
print(f'(3) MC sigma / empirical sigma: median {np.median(ratio):.2f}')

x = rng.normal(chi_meas_syn, 0.02); c = rng.normal(chi_cal_syn, 0.02)
print('max |ours - challenge_utils.postselection_and_mitigation| =',
      np.nanmax(np.abs(odr_mitigate(x, c, chi_exact_syn, L) - cu.postselection_and_mitigation(x, c, chi_exact_syn, L))))

(1) noise-free recovery: max |error| = 4.4e-16, NaN sites = [5, 62]
(2) bias formula exact to 4.0e-16
(3) MC sigma / empirical sigma: median 1.01
max |ours - challenge_utils.postselection_and_mitigation| = 0.0


## Ex 3.2 — Runtime mitigation options

In [7]:
def mitigation_options(base_options, num_randomizations, shots_per_randomization, dd_sequence="XY4", readout_mitigation=False):
    """Runtime options of one hardware job: the Part-2 options + a twirling budget + a DD sequence (+ TREX for the improvement run).
    ODR itself stays ours: the ODR run is submitted raw (resilience_level 0, no server-side mitigation)."""
    opts = copy.deepcopy(base_options)
    settings = {
        "twirling": {"enable_gates": True, "enable_measure": True, "num_randomizations": num_randomizations,
                     "shots_per_randomization": shots_per_randomization, "strategy": "active-accum"},
        "dynamical_decoupling": {"enable": True, "sequence_type": dd_sequence},
        "resilience": {"measure_mitigation": readout_mitigation, "zne_mitigation": False, "pec_mitigation": False},
    }
    if isinstance(opts, dict):
        opts["resilience_level"] = 0
        for group, fields in settings.items():
            if not isinstance(opts.get(group), dict):
                opts[group] = {}
            opts[group].update(fields)
    else:
        opts.resilience_level = 0
        for group, fields in settings.items():
            sub = getattr(opts, group)
            for key, value in fields.items():
                setattr(sub, key, value)
    return opts

In [8]:
from qiskit_ibm_runtime import EstimatorV2 as RuntimeEstimatorV2
from qiskit_ibm_runtime.options import EstimatorOptions

base = {'max_execution_time': 180, 'twirling': {'enable_gates': False}}
before = copy.deepcopy(base)
print('dict form :', mitigation_options(base, 64, 256, 'XpXm', readout_mitigation=True))
assert base == before, 'base options must not be modified'

eo = EstimatorOptions(); eo.max_execution_time = 180
o = mitigation_options(eo, 64, 256, 'XpXm')
print('object form:', o.twirling, o.dynamical_decoupling.sequence_type, o.resilience_level, o.max_execution_time)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    RuntimeEstimatorV2(mode=backend, options=o)
print('accepted by qiskit_ibm_runtime.EstimatorV2')

dict form : {'max_execution_time': 180, 'twirling': {'enable_gates': True, 'enable_measure': True, 'num_randomizations': 64, 'shots_per_randomization': 256, 'strategy': 'active-accum'}, 'resilience_level': 0, 'dynamical_decoupling': {'enable': True, 'sequence_type': 'XpXm'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}}
object form: TwirlingOptions(enable_gates=True, enable_measure=True, num_randomizations=64, shots_per_randomization=256, strategy='active-accum') XpXm 0 180
accepted by qiskit_ibm_runtime.EstimatorV2
